# 🛒 Python Web Scraping Project

Welcome to the **E-Commerce Web Scraping** project!

### 🎯 Project Goal
Scrape **50 unique products** from [Books to Scrape](http://books.toscrape.com/) and save them directly into a CSV file.

### 📊 Data Fields Collected
1. `product_name`: Full title of the book
2. `price`: Price of the product
3. `rating`: Star rating
4. `availability`: Stock availability (e.g., In stock (22 available))
5. `number_of_reviews`: Review count
6. `category`: Book genre from breadcrumb hierarchy
7. `product_url`: Absolute URL to the product page

## 1. Imports & Configuration

In [ ]:
import time
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Global Configuration
BASE_URL = "http://books.toscrape.com/catalogue/"
START_URL = "http://books.toscrape.com/catalogue/page-1.html"
TARGET_COUNT = 50
REQUEST_DELAY = 0.05  # Polite delay between requests (seconds)
OUTPUT_FILE = "products.csv"
RAW_OUTPUT_FILE = "products_raw.csv"

# Custom HTTP headers to simulate standard browser traffic
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

print("Configuration loaded successfully: Target = 50 products.")

## 2. HTTP Request Handler (`get_page`)

In [ ]:
def get_page(url: str, session: requests.Session = None, retries: int = 3, delay: float = REQUEST_DELAY) -> BeautifulSoup:
    """
    Sends an HTTP GET request to the given URL and returns parsed BeautifulSoup HTML.
    Includes error handling and automatic retries.
    """
    http = session or requests
    for attempt in range(1, retries + 1):
        try:
            if delay > 0:
                time.sleep(delay)
            response = http.get(url, headers=HEADERS, timeout=10)
            response.raise_for_status()
            response.encoding = response.apparent_encoding or "utf-8"
            return BeautifulSoup(response.text, "html.parser")
        except requests.exceptions.RequestException as e:
            print(f"[Warning] Failed to fetch {url} (Attempt {attempt}/{retries}): {e}")
            if attempt < retries:
                time.sleep(0.5 * attempt)
            else:
                print(f"[Error] Could not retrieve URL after {retries} attempts: {url}")
                return None

## 3. Product Extraction Logic (`scrape_product`)

In [ ]:
def scrape_product(pod: BeautifulSoup, session: requests.Session = None, base_url: str = BASE_URL) -> dict:
    """
    Extracts product fields from a catalog card and visits the detail page for Category, Availability, and Reviews.
    """
    try:
        # 1. Product Name (full title attribute from the <a> tag inside <h3>)
        title_tag = pod.select_one("h3 a")
        product_name = title_tag.get("title", "").strip() if title_tag else None
        if not product_name and title_tag:
            product_name = title_tag.get_text(strip=True)

        # 2. Product URL (resolved relative link)
        relative_link = title_tag.get("href", "") if title_tag else ""
        product_url = urljoin(base_url, relative_link)

        # 3. Price (raw text)
        price_tag = pod.select_one("p.price_color")
        price = price_tag.get_text(strip=True) if price_tag else None

        # 4. Rating (CSS class name: 'star-rating Three' -> 'Three')
        rating_tag = pod.select_one("p.star-rating")
        rating = None
        if rating_tag:
            rating_classes = [c for c in rating_tag.get("class", []) if c != "star-rating"]
            rating = rating_classes[0] if rating_classes else None

        # 5. Detail Page Attributes: Category, Availability & Number of Reviews
        category = "Unknown"
        availability = "Unknown"
        num_reviews = "0"

        if product_url:
            detail_soup = get_page(product_url, session=session, delay=REQUEST_DELAY)
            if detail_soup:
                # Category from breadcrumbs (Home > Books > [Category] > Title)
                breadcrumb_items = detail_soup.select("ul.breadcrumb li")
                if len(breadcrumb_items) >= 3:
                    category = breadcrumb_items[2].get_text(strip=True)

                # Availability & Number of Reviews from the Product Information table
                info_table = detail_soup.select_one("table.table-striped")
                if info_table:
                    for row in info_table.select("tr"):
                        th_text = row.th.get_text(strip=True) if row.th else ""
                        if "Availability" in th_text:
                            availability = row.td.get_text(strip=True) if row.td else "Unknown"
                        elif "Number of reviews" in th_text:
                            num_reviews = row.td.get_text(strip=True) if row.td else "0"

        return {
            "product_name": product_name,
            "price": price,
            "rating": rating,
            "availability": availability,
            "number_of_reviews": num_reviews,
            "category": category,
            "product_url": product_url
        }
    except Exception as err:
        print(f"[Warning] Error parsing product pod: {err}")
        return None

## 4. Catalog Page Processor (`scrape_page`)

In [ ]:
def scrape_page(soup: BeautifulSoup, session: requests.Session, collected_data: list, 
                collected_urls: set, target_count: int = TARGET_COUNT) -> int:
    """
    Finds and processes all product pods on a given catalog page.
    Appends unique items to collected_data until target_count is reached.
    """
    pods = soup.select("article.product_pod")
    for pod in pods:
        if len(collected_data) >= target_count:
            break

        # Check URL uniqueness before deep scraping detail page
        title_tag = pod.select_one("h3 a")
        rel_link = title_tag.get("href", "") if title_tag else ""
        prod_url = urljoin(BASE_URL, rel_link)

        if prod_url in collected_urls:
            continue  # Avoid duplicate scrape

        product_data = scrape_product(pod, session=session)
        if product_data and product_data.get("product_name"):
            collected_data.append(product_data)
            collected_urls.add(product_data["product_url"])
            if len(collected_data) % 10 == 0 or len(collected_data) == target_count:
                print(f"Scraped {len(collected_data)}/{target_count} products")

    return len(collected_data)

## 5. Saving Utilities & Main Flow (`save_data` & `main`)

In [ ]:
def save_data(df: pd.DataFrame, filename: str) -> None:
    """
    Saves DataFrame to CSV file without index.
    """
    df.to_csv(filename, index=False, encoding="utf-8")
    print(f"[Success] Saved {len(df)} records to '{filename}'")


def main():
    """
    Main execution flow.
    """
    print("=" * 60)
    print(f"Starting Scraper: Target = {TARGET_COUNT} Products")
    print("=" * 60)

    session = requests.Session()
    collected_data = []
    collected_urls = set()
    page_number = 1

    start_time = time.time()

    # Pagination Loop
    while len(collected_data) < TARGET_COUNT:
        page_url = f"http://books.toscrape.com/catalogue/page-{page_number}.html"
        print(f"\n[Page {page_number}] Fetching catalog page: {page_url}")
        
        soup = get_page(page_url, session=session)
        if not soup:
            print(f"[Info] Could not fetch page {page_number}. Ending pagination.")
            break

        pods = soup.select("article.product_pod")
        if not pods:
            print(f"[Info] No more products found on page {page_number}. Ending pagination.")
            break

        scrape_page(soup, session, collected_data, collected_urls, target_count=TARGET_COUNT)

        next_btn = soup.select_one("li.next a")
        if not next_btn and len(collected_data) < TARGET_COUNT:
            print("[Info] No next page available.")
            break

        page_number += 1

    elapsed_time = round(time.time() - start_time, 2)
    print("\n" + "=" * 60)
    print(f"Scraping Finished! Total Products Scraped: {len(collected_data)} in {elapsed_time}s")
    print("=" * 60)

    # Convert to DataFrame
    df = pd.DataFrame(collected_data)

    # Save Data directly
    save_data(df, OUTPUT_FILE)
    save_data(df, RAW_OUTPUT_FILE)
    
    return df

# Run the scraper
df = main()

## 6. Dataset Preview

In [ ]:
# Preview first 5 rows
df.head()

In [ ]:
# Dataset information
df.info()